# Polymarket Trading Bot Demo

This notebook demonstrates how to use the Polymarket Trading Bot to:
1. Fetch market data from Polymarket
2. Analyze probabilities and sentiment
3. Generate trading signals
4. Manage positions and risk

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))

import yaml
import json
from pprint import pprint

## 1. Fetch Market Data

In [ ]:
from market_data import PolymarketDataFetcher

# Initialize data fetcher
fetcher = PolymarketDataFetcher(rate_limit_delay=0.6)

# Get active markets
markets = fetcher.get_all_markets(limit=10, active=True)

print(f"Found {len(markets)} active markets\n")
print("First 5 markets:")
for i, market in enumerate(markets[:5], 1):
    print(f"{i}. {market.get('question', 'N/A')[:80]}")

## 2. Analyze Market Sentiment

In [ ]:
# Get detailed sentiment for first market
if markets:
    market_id = markets[0].get('id')
    sentiment = fetcher.get_market_sentiment(market_id)
    
    print(f"Market: {sentiment.get('question')}\n")
    print("Outcomes and Probabilities:")
    for outcome in sentiment.get('outcomes', []):
        print(f"  {outcome['outcome']}: {outcome['implied_probability']:.2%} (Price: {outcome['price']:.4f})")

## 3. Generate Trading Signals

In [ ]:
from strategy import StrategyManager

# Load configuration
config_path = Path.cwd().parent / 'config' / 'config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Initialize strategy manager
strategy_manager = StrategyManager(config)

# Generate signals for the market
if sentiment:
    all_signals = strategy_manager.generate_all_signals(sentiment)
    
    print(f"Generated {len(all_signals)} strategy signals:\n")
    for strategy_signal in all_signals:
        print(f"Strategy: {strategy_signal['strategy']}")
        print(f"Signals: {len(strategy_signal['signals'])}")
        for signal in strategy_signal['signals']:
            print(f"  - {signal['signal']}: {signal['outcome']} (Confidence: {signal['confidence']:.2%})")
            print(f"    {signal['reasoning']}")
        print()

## 4. Risk Management and Position Sizing

In [ ]:
from risk_manager import RiskManager

# Initialize risk manager
risk_manager = RiskManager(config.get('risk_management', {}))

# Get best signal
if all_signals:
    best_signal = strategy_manager.get_best_signal(all_signals)
    
    if best_signal:
        print("Best Trading Signal:")
        print(f"  Signal: {best_signal['signal']}")
        print(f"  Outcome: {best_signal['outcome']}")
        print(f"  Confidence: {best_signal['confidence']:.2%}")
        print(f"  Edge: {best_signal['edge']:.2%}")
        print(f"  Reasoning: {best_signal['reasoning']}")
        
        # Calculate position size
        position_size = risk_manager.calculate_position_size(best_signal)
        print(f"\nRecommended Position Size: ${position_size:.2f}")

## 5. Run Trading Bot (Single Cycle)

In [ ]:
from bot import PolymarketTradingBot

# Initialize bot
bot = PolymarketTradingBot()

# Run a single trading cycle
bot.run_single_cycle()

## 6. Portfolio Analysis

In [ ]:
# Get portfolio summary
summary = bot.risk_manager.get_portfolio_summary()

print("Portfolio Summary:")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key}: ${value:,.2f}" if 'pct' not in key else f"  {key}: {value:.2f}%")
    else:
        print(f"  {key}: {value}")

# Get active positions
positions = bot.risk_manager.get_position_details()
print(f"\nActive Positions: {len(positions)}")
for pos in positions:
    print(f"  {pos['outcome']}: {pos['side']} ${pos['size']:.2f} @ {pos['entry_price']:.4f}")

## 7. Find Trading Opportunities

In [ ]:
# Scan for trading opportunities
opportunities = fetcher.find_trading_opportunities(markets, threshold=0.05)

print(f"Found {len(opportunities)} trading opportunities:\n")
for i, opp in enumerate(opportunities[:5], 1):
    print(f"{i}. {opp['question'][:60]}")
    print(f"   Type: {opp.get('opportunity_type')}")
    if 'deviation' in opp:
        print(f"   Probability sum: {opp['probability_sum']:.2%} (Deviation: {opp['deviation']:.2%})")
    elif 'probability' in opp:
        print(f"   {opp['outcome']}: {opp['probability']:.2%}")
    print()

## 8. Custom Strategy Testing

In [ ]:
from strategy import MeanReversionStrategy, ArbitrageStrategy

# Test mean reversion strategy
mr_config = {
    'min_edge': 0.03,
    'confidence_threshold': 0.6,
    'extreme_low': 0.10,
    'extreme_high': 0.90
}

mr_strategy = MeanReversionStrategy(mr_config)

if sentiment:
    mr_signals = mr_strategy.generate_signal(sentiment)
    print("Mean Reversion Strategy Signals:")
    pprint(mr_signals)